In [ ]:
pip install requests beautifulsoup4

In [ ]:
# the necessary imports
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re

In [22]:
oscars = pd.read_csv(
    "https://raw.githubusercontent.com/DLu/oscar_data/main/oscars.csv",
    sep="\t",
    encoding="utf-8"
)
print(oscars.shape)
oscars.head()

(12118, 14)


,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins|The Patent Leather Kid,NaN,NaN
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexand...,NaN,NaN
2,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN
3,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven|Street Angel|Sunrise,tt0018379|tt0019429|tt0018455,Janet Gaynor,Janet Gaynor,nm0310980,True,Diane|Angela|The Wife,NaN,NaN
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,nm0841797,NaN,Sadie Thompson,NaN,NaN


In [ ]:
# Clean up oscars.csv 
bp['Winner'] = bp['Winner'].fillna(False)
def make_slug(title):
    """Convert a film title to The Numbers URL slug."""
    slug = str(title).strip()
    slug = re.sub(r"[''']", "",  slug)       
    slug = re.sub(r"[:&,!?.]", "", slug)     
    slug = re.sub(r"\s+", "-", slug)         
    return slug

bp['slug'] = bp['Film'].apply(make_slug)
print(bp[['Film', 'slug']].head(20).to_string())
bp_modern = bp[bp['Ceremony'] >= 73].copy()

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (academic research project)"}
BASE_URL = "https://www.the-numbers.com/movie/"
def scrape_film(slug):
    url = f"{BASE_URL}{slug}?public=true"
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return None, url, r.status_code
        
        soup = BeautifulSoup(r.text, "html.parser")
        data = {"url": url}
        
        # The Numbers puts financials in a <div id="summary"> table
        summary = soup.find("div", id="summary")
        if summary:
            for row in summary.find_all("tr"):
                cells = row.find_all("td")
                if len(cells) == 2:
                    label = cells[0].get_text(strip=True).rstrip(":")
                    value = cells[1].get_text(strip=True)
                    data[label] = value
        
        # Also grab the movie title from the page to confirm we got the right film
        h1 = soup.find("h1")
        data["page_title"] = h1.get_text(strip=True) if h1 else ""
        
        return data, url, r.status_code

    except Exception as e:
        return None, url, str(e)


# ── Main loop ─────────────────────────────────────────────────────────────────
results = []
failed = []

for i, row in bp_modern.iterrows():
    film  = row['Film']
    year  = row['Year']
    slug  = row['slug']
    
    print(f"  [{i}] {film}...", end=" ")
    data, url, status = scrape_film(slug)
    
    if data:
        data['Film']      = film
        data['Year']      = year
        data['Ceremony']  = row['Ceremony']
        data['Winner']    = row['Winner']
        data['FilmId']    = row['FilmId']
        results.append(data)
        print(f"OK ({status})")
    else:
        failed.append({'Film': film, 'Year': year, 'slug': slug, 
                       'url': url, 'status': status})
        print(f"FAILED ({status})")
    
    time.sleep(2)  # polite pause

print(f"\nDone. Succeeded: {len(results)}, Failed: {len(failed)}")